# 01 - Extract Hard Negatives (3-slice v2)

Variante v2 : patches 3 canaux `(slice-1, slice, slice+1)` avec `crop_margin=30` (était 10 en v1) pour exposer le contexte pancréatique. Sortie : `outputs/msd_implementation/resnet50_wide_crop/datasets/classifier_dataset_resnet50_wide_crop/`.

In [ ]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "temp"
PROJECT_DIR = Path("/content/INF8225_Projet")
DRIVE_FOLDER = None
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from colab.setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())

In [ ]:
#@title Verify shared assets
from pathlib import Path

required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("data/MSD_pancreas/train/images"),
    Path("data/MSD_pancreas/val/images"),
    Path("data/MSD_pancreas/test/images"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("msd_implementation/configs/grounding_dino/pancreas_tumor.py"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"

In [ ]:
#@title Run pipeline step (3-slice v2 extraction)
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet50_wide_crop.extract_hard_negatives"], check=True)

In [ ]:
#@title Inspect generated 3-slice v2 classifier dataset
from pathlib import Path
base = Path("outputs/msd_implementation/resnet50_wide_crop/datasets/classifier_dataset_resnet50_wide_crop")
for split in ["train", "val"]:
    for cls in ["0", "1"]:
        folder = base / split / cls
        count = len(list(folder.glob("*.png"))) if folder.exists() else 0
        print(f"{split}/{cls}: {count} patches")
assert (base / "train" / "0").exists(), "classifier_dataset_3slice_v2 was not created"